# TP1 Machine Learning
## Système Expert pour la résolution des Tours de Hanoï

- **Realisé Par : Houda HAJJAJ et Abdelfatah CHAIB**
- **M2 EUR Smart IoT** 

### Objectif du TP

L’objectif de ce TP est de développer un système expert capable de résoudre le problème des Tours de Hanoï à l’aide d’une base de connaissances composée de règles, d’une métarègle de priorité et d’un moteur d’inférence par chaînage avant.

## Schéma général de la solution

```text
┌─────────────────────────────────────────────┐
│          INITIALISATION DU JEU              │
│                                             │
│  - Création de Jeu_Hanoi                    │
│  - 3 pics                                   │
│  - 3 palets                                 │
│  - Situation initiale                       │
└──────────────────────┬──────────────────────┘
                       ▼
┌─────────────────────────────────────────────┐
│       REPRÉSENTATION DE LA SITUATION        │
│                                             │
│  jeu.pic                                    │
│  jeu.nombre_palet                           │
└──────────────────────┬──────────────────────┘
                       ▼
┌─────────────────────────────────────────────┐
│       IDENTIFICATION DE LA SITUATION        │
│                                             │
│  nombre_situation(jeu)                      │
│  → transforme une situation en identifiant  │
└──────────────────────┬──────────────────────┘
                       ▼
┌─────────────────────────────────────────────┐
│       VÉRIFICATION DES CONDITIONS           │
│                                             │
│  pic_vide()                                 │
│  regle_jeu()                                │
│  situation_vue()                            │
└──────────────────────┬──────────────────────┘
                       ▼
┌─────────────────────────────────────────────┐
│             BASE EXPERTE                    │
│                                             │
│  Règle 1 : Pic 0 → Pic 1                    │
│  Règle 2 : Pic 1 → Pic 2                    │
│  Règle 3 : Pic 2 → Pic 1                    │
│  Règle 4 : Pic 1 → Pic 0                    │
└──────────────────────┬──────────────────────┘
                       ▼
┌─────────────────────────────────────────────┐
│              MÉTARÈGLE                      │
│                                             │
│  Plusieurs règles possibles ?               │
│  → choisir la règle de plus petit indice    │
└──────────────────────┬──────────────────────┘
                       ▼
┌─────────────────────────────────────────────┐
│          MOTEUR D'INFÉRENCE                 │
│                                             │
│  - tester les règles                        │
│  - sélectionner une règle                   │
│  - effectuer le déplacement                 │
│  - mémoriser la nouvelle situation          │
│  - incrémenter le nombre de coups           │
└──────────────────────┬──────────────────────┘
                       ▼
              ┌───────────────────┐
              │ Situation finale ?│
              └─────────┬─────────┘
                        │
                  Non ──┘ ──► Nouvelle itération
                        │
                       Oui
                        ▼
┌─────────────────────────────────────────────┐
│              ANALYSE DU RÉSULTAT            │
│                                             │
│  - nombre de coups obtenu                   │
│  - comparaison avec l'optimum = 7           │
│  - analyse de la stratégie                  │
│  - éventuelles améliorations                │
└─────────────────────────────────────────────┘


Le schéma résume bien le principe du **chaînage avant** : partir de la situation initiale, appliquer une règle, obtenir une nouvelle situation et recommencer jusqu'à la situation finale.

## 1. Importation et préparation

In [1]:
import numpy as np
import copy

## 2. Représentation du jeu

Nous définissons la classe `Jeu_Hanoi` permettant de représenter l'état du problème.  
Le jeu contient trois pics et trois positions possibles par pic.

Deux structures sont utilisées :

- `jeu.pic` : représente les palets présents sur chaque pic ;
- `jeu.nombre_palet` : indique le nombre de palets présents sur chaque pic.

In [2]:
class Jeu_Hanoi:
    def __init__(self):
        self.pic = np.zeros([3, 3], dtype=int)
        self.nombre_palet = np.zeros(3, dtype=int)
        pass

## 3. Initialisation de la situation initiale

Nous créons une instance du jeu et plaçons les trois palets sur le pic de départ.

La situation initiale est :

- Pic 0 : `[3, 2, 1]`
- Pic 1 : `[0, 0, 0]`
- Pic 2 : `[0, 0, 0]`

Le tableau `nombre_palet` vaut alors `[3, 0, 0]`.

In [3]:
jeu = Jeu_Hanoi()

jeu.nombre_palet[0] = 3

jeu.pic[0, 0] = 3
jeu.pic[0, 1] = 2
jeu.pic[0, 2] = 1

In [4]:
jeu.pic

array([[3, 2, 1],
       [0, 0, 0],
       [0, 0, 0]])

In [5]:
jeu.nombre_palet

array([3, 0, 0])

## 4. Identification d'une situation

Pour permettre au système expert de reconnaître les situations déjà rencontrées, chaque configuration du jeu est transformée en un nombre unique.

La fonction `nombre_situation(jeu)` parcourt les trois pics et utilise une représentation basée sur des puissances de 2.

Pour chaque palet de taille `taille` situé sur le pic `b`, on ajoute :

$$
2^{(\text{taille}-1)+3b}
$$

Cette représentation permet ensuite de mémoriser les situations et d'éviter les cycles.

In [6]:
def nombre_situation(jeu):
    nombre = 0

    # Première boucle : parcourir les 3 pics
    for b in range(3):

        # Deuxième boucle : parcourir les palets présents sur le pic
        for palet in range(jeu.nombre_palet[b]):

            # Taille du disque
            taille = jeu.pic[b, palet]

            # Chaque disque correspond à une puissance de 2
            # avec un décalage de b * 3 selon le pic
            nombre += 2 ** (taille - 1 + b * 3)

    return nombre

In [7]:
print(nombre_situation(jeu))

7


## 5. Vérification de l'état d'un pic

La fonction `pic_vide(indice_pic, jeu)` permet de déterminer si un pic ne contient aucun palet.

Elle utilise le tableau `nombre_palet` :

$$
\text{pic vide} \iff \text{nombre\_palet}[indice\_pic] = 0
$$

Cette fonction sera utilisée par les règles pour déterminer si un déplacement peut être envisagé.

In [8]:
def pic_vide(indice_pic, jeu):
    reponse = jeu.nombre_palet[indice_pic] == 0
    return reponse

In [9]:
print(pic_vide(0, jeu))
print(pic_vide(1, jeu))
print(pic_vide(2, jeu))

False
True
True


## 6. Vérification de la validité d'un déplacement

La fonction `regle_jeu(indice_pic1, indice_pic2, jeu)` permet de déterminer si un déplacement d'un palet du pic `indice_pic1` vers le pic `indice_pic2` respecte les règles des Tours de Hanoï.

Un déplacement est autorisé si :

1. le pic de départ contient au moins un palet ;
2. le pic d'arrivée est vide ; ou
3. le palet supérieur du pic de départ est plus petit que le palet supérieur du pic d'arrivée.

La fonction retourne un booléen indiquant si le déplacement est autorisé.

In [10]:
def regle_jeu(indice_pic1, indice_pic2, jeu):
    # Vérifier que le pic de départ contient au moins un palet
    if jeu.nombre_palet[indice_pic1] == 0:
        return False

    # Si le pic d'arrivée est vide, le déplacement est autorisé
    if jeu.nombre_palet[indice_pic2] == 0:
        return True

    # Récupérer le palet supérieur du pic de départ
    palet_depart = jeu.pic[
        indice_pic1,
        jeu.nombre_palet[indice_pic1] - 1
    ]

    # Récupérer le palet supérieur du pic d'arrivée
    palet_arrivee = jeu.pic[
        indice_pic2,
        jeu.nombre_palet[indice_pic2] - 1
    ]

    # Le palet déplacé doit être plus petit
    if palet_depart < palet_arrivee:
        return True

    return False

In [11]:
# Test 1 : déplacer un disque vers un pic vide
print("Test 1 :", regle_jeu(0, 1, jeu))
# Résultat attendu : True


# Test 2 : déplacer un petit disque sur un grand disque
jeu_test = Jeu_Hanoi()

jeu_test.nombre_palet[0] = 1
jeu_test.pic[0, 0] = 1

jeu_test.nombre_palet[1] = 1
jeu_test.pic[1, 0] = 3

print("Test 2 :", regle_jeu(0, 1, jeu_test))
# Résultat attendu : True


# Test 3 : essayer de déplacer un grand disque sur un petit disque
jeu_test = Jeu_Hanoi()

jeu_test.nombre_palet[0] = 1
jeu_test.pic[0, 0] = 3

jeu_test.nombre_palet[1] = 1
jeu_test.pic[1, 0] = 1

print("Test 3 :", regle_jeu(0, 1, jeu_test))
# Résultat attendu : False


# Test 4 : essayer de déplacer depuis un pic vide
jeu_test = Jeu_Hanoi()

print("Test 4 :", regle_jeu(0, 1, jeu_test))
# Résultat attendu : False

Test 1 : True
Test 2 : True
Test 3 : False
Test 4 : False


## 7. Exécution d’un déplacement

Cette fonction réalise effectivement le déplacement d’un palet du pic de départ vers le pic d’arrivée. Elle met à jour à la fois la matrice `pic` et le tableau `nombre_palet`. Le nombre total de palets doit rester égal à 3.

In [12]:
def effectue_deplacement(indice_pic1, indice_pic2, jeu):

    # Position du palet supérieur sur le pic de départ
    position_depart = jeu.nombre_palet[indice_pic1] - 1

    # Récupérer le palet à déplacer
    palet = jeu.pic[indice_pic1, position_depart]

    # Retirer le palet du pic de départ
    jeu.pic[indice_pic1, position_depart] = 0
    jeu.nombre_palet[indice_pic1] -= 1

    # Position où placer le palet sur le pic d'arrivée
    position_arrivee = jeu.nombre_palet[indice_pic2]

    # Placer le palet sur le pic d'arrivée
    jeu.pic[indice_pic2, position_arrivee] = palet
    jeu.nombre_palet[indice_pic2] += 1

### Test de la fonction `effectue_deplacement()`

On teste ici un déplacement du palet supérieur du **pic 0 vers le pic 1**. On vérifie également que le nombre total de palets reste égal à 3.

In [13]:
jeu_test = Jeu_Hanoi()

jeu_test.nombre_palet[0] = 3
jeu_test.pic[0, 0] = 3
jeu_test.pic[0, 1] = 2
jeu_test.pic[0, 2] = 1

print("Avant déplacement :")
print(jeu_test.pic)
print(jeu_test.nombre_palet)

effectue_deplacement(0, 1, jeu_test)

print("\nAprès déplacement :")
print(jeu_test.pic)
print(jeu_test.nombre_palet)

print("\nNombre total de palets :",
      np.sum(jeu_test.nombre_palet))

Avant déplacement :
[[3 2 1]
 [0 0 0]
 [0 0 0]]
[3 0 0]

Après déplacement :
[[3 2 0]
 [1 0 0]
 [0 0 0]]
[2 1 0]

Nombre total de palets : 3


## 8. Vérification d’une nouvelle situation

Cette fonction permet de vérifier si un déplacement conduirait à **une situation déjà étudiée**. Pour cela, on crée une copie du jeu, on effectue le déplacement sur cette copie et on calcule son identifiant avec `nombre_situation()`.

La situation réelle `jeu` ne doit jamais être modifiée pendant cette vérification.

In [14]:
def situation_vue(indice_pic1, indice_pic2, jeu, situation_etudiee):

    # Vérifier que le déplacement est possible
    if not regle_jeu(indice_pic1, indice_pic2, jeu):
        return False

    # Créer une copie indépendante du jeu
    jeu_tmp = Jeu_Hanoi()

    jeu_tmp.pic = copy.deepcopy(jeu.pic)
    jeu_tmp.nombre_palet = copy.deepcopy(jeu.nombre_palet)

    # Simuler le déplacement sur la copie
    effectue_deplacement(indice_pic1, indice_pic2, jeu_tmp)

    # Calculer l'identifiant de la nouvelle situation
    nombre_nouvelle_situation = nombre_situation(jeu_tmp)

    # Vérifier si cette situation est nouvelle
    reponse = nombre_nouvelle_situation not in situation_etudiee

    return reponse

### Test de la fonction `situation_vue()`

On commence avec la situation initiale. L'identifiant de la situation initiale est ajouté à la liste des situations étudiées. On vérifie ensuite que le déplacement `0 → 1` produit une nouvelle situation et que le jeu original n'est pas modifié.

In [15]:
jeu_test = Jeu_Hanoi()

jeu_test.nombre_palet[0] = 3
jeu_test.pic[0, 0] = 3
jeu_test.pic[0, 1] = 2
jeu_test.pic[0, 2] = 1

# Situation initiale
situation_initiale = nombre_situation(jeu_test)

# Liste des situations déjà étudiées
situation_etudiee = [situation_initiale]

print("Situation initiale :", situation_initiale)

# Vérification du déplacement 0 -> 1
print("Nouvelle situation :", 
      situation_vue(0, 1, jeu_test, situation_etudiee))

# Vérifier que le jeu original n'a pas été modifié
print("\nJeu original après la vérification :")
print(jeu_test.pic)
print(jeu_test.nombre_palet)

Situation initiale : 7
Nouvelle situation : True

Jeu original après la vérification :
[[3 2 1]
 [0 0 0]
 [0 0 0]]
[3 0 0]


## 9. Base experte : Définition des règles

La base experte contient les quatre règles de déplacement. Chaque règle indique le pic de départ et le pic d’arrivée. Une règle ne sera appliquée que si le déplacement est légal et conduit à une nouvelle situation.

In [16]:
# Définition des quatre règles
regles = [
    (0, 1),  # Règle 1 : pic 0 -> pic 1
    (1, 2),  # Règle 2 : pic 1 -> pic 2
    (2, 1),  # Règle 3 : pic 2 -> pic 1
    (1, 0)   # Règle 4 : pic 1 -> pic 0
]

### Test de la base experte
On affiche simplement les quatre règles pour vérifier leur définition.

In [17]:
for i, regle in enumerate(regles, start=1):
    print(f"Règle {i} : pic {regle[0]} -> pic {regle[1]}")

Règle 1 : pic 0 -> pic 1
Règle 2 : pic 1 -> pic 2
Règle 3 : pic 2 -> pic 1
Règle 4 : pic 1 -> pic 0


## 10. Métarègle : Priorité des règles

La métarègle permet de choisir une règle lorsqu’il existe plusieurs règles applicables. On impose de choisir la règle ayant le plus petit indice.
Ainsi, les règles seront toujours testées dans l’ordre :

`Règle 1 → Règle 2 → Règle 3 → Règle 4`

Dès qu’une règle est éligible, elle est sélectionnée et les règles suivantes ne sont pas examinées pour cette itération.

Cette priorité peut donc être respectée simplement en parcourant `regles` dans son ordre naturel.

## 11. Vérification de l’éligibilité d’une règle
Une règle est éligible si :

- le pic de départ contient un palet ;
- le déplacement respecte regle_jeu() ;
- la situation obtenue n’a pas encore été étudiée.

In [18]:
def regle_eligible(indice_pic1, indice_pic2, jeu, situation_etudiee):

    # Vérifier que le pic de départ n'est pas vide
    if jeu.nombre_palet[indice_pic1] == 0:
        return False

    # Vérifier que le déplacement est légal
    if not regle_jeu(indice_pic1, indice_pic2, jeu):
        return False

    # Vérifier que la nouvelle situation est inconnue
    if not situation_vue(indice_pic1, indice_pic2, jeu, situation_etudiee):
        return False

    return True

### Test de l’éligibilité d’une règle

On repart de la situation initiale et on vérifie la première règle.

In [19]:
jeu_test = Jeu_Hanoi()

jeu_test.nombre_palet[0] = 3
jeu_test.pic[0, 0] = 3
jeu_test.pic[0, 1] = 2
jeu_test.pic[0, 2] = 1

situation_initiale = nombre_situation(jeu_test)
situation_etudiee = [situation_initiale]

print("Règle 1 éligible :",
      regle_eligible(0, 1, jeu_test, situation_etudiee))

Règle 1 éligible : True


## 12. Moteur d’inférence en chaînage avant
Le moteur d’inférence applique progressivement les règles à partir de la situation initiale. À chaque étape, les règles sont testées dans l’ordre de priorité. La première règle éligible est appliquée, la nouvelle situation est mémorisée, puis le processus recommence jusqu’à atteindre la situation finale.

In [20]:
# Initialisation du jeu
jeu = Jeu_Hanoi()

jeu.nombre_palet[0] = 3
jeu.pic[0, 0] = 3
jeu.pic[0, 1] = 2
jeu.pic[0, 2] = 1

# Calcul de la situation initiale
situation_initiale = nombre_situation(jeu)

# Liste des situations déjà étudiées
situation_etudiee = [situation_initiale]

# Construction de la situation finale
jeu_final = Jeu_Hanoi()

jeu_final.nombre_palet[2] = 3
jeu_final.pic[2, 0] = 3
jeu_final.pic[2, 1] = 2
jeu_final.pic[2, 2] = 1

# Calcul de la situation finale
situation_finale = nombre_situation(jeu_final)

# Compteur de coups
nombre_coup = 0

print("Situation initiale :", situation_initiale)
print("Situation finale :", situation_finale)

Situation initiale : 7
Situation finale : 448


## 13. Application des règles par le moteur

On parcourt les quatre règles dans leur ordre de priorité. Dès qu'une règle est éligible, elle est appliquée. L'identifiant de la nouvelle situation est ensuite ajouté à `situation_etudiee`.

In [21]:
while nombre_situation(jeu) != situation_finale:

    regle_appliquee = False

    # Tester les règles dans l'ordre de priorité
    for indice_regle, regle in enumerate(regles, start=1):

        indice_pic1 = regle[0]
        indice_pic2 = regle[1]

        # Vérifier si la règle est éligible
        if regle_eligible(
            indice_pic1,
            indice_pic2,
            jeu,
            situation_etudiee
        ):

            # Effectuer le déplacement
            effectue_deplacement(
                indice_pic1,
                indice_pic2,
                jeu
            )

            # Calculer la nouvelle situation
            nouvelle_situation = nombre_situation(jeu)

            # Mémoriser la nouvelle situation
            situation_etudiee.append(nouvelle_situation)

            # Incrémenter le nombre de coups
            nombre_coup += 1

            # Afficher le numéro du coup
            print(f"Coup {nombre_coup}")

            # Afficher la règle sélectionnée
            print(
                f"Règle {indice_regle} : "
                f"pic {indice_pic1} -> pic {indice_pic2}"
            )

            # Afficher l'état obtenu
            print("Nouvelle situation :", nouvelle_situation)
            print(jeu.pic)
            print()

            regle_appliquee = True

            # Une seule règle est appliquée par itération
            break

    # Aucune règle applicable
    if not regle_appliquee:
        print("Aucune règle applicable.")
        break

Coup 1
Règle 1 : pic 0 -> pic 1
Nouvelle situation : 14
[[3 2 0]
 [1 0 0]
 [0 0 0]]

Coup 2
Règle 2 : pic 1 -> pic 2
Nouvelle situation : 70
[[3 2 0]
 [0 0 0]
 [1 0 0]]

Coup 3
Règle 1 : pic 0 -> pic 1
Nouvelle situation : 84
[[3 0 0]
 [2 0 0]
 [1 0 0]]

Coup 4
Règle 3 : pic 2 -> pic 1
Nouvelle situation : 28
[[3 0 0]
 [2 1 0]
 [0 0 0]]

Coup 5
Règle 4 : pic 1 -> pic 0
Nouvelle situation : 21
[[3 1 0]
 [2 0 0]
 [0 0 0]]

Coup 6
Règle 2 : pic 1 -> pic 2
Nouvelle situation : 133
[[3 1 0]
 [0 0 0]
 [2 0 0]]

Coup 7
Règle 1 : pic 0 -> pic 1
Nouvelle situation : 140
[[3 0 0]
 [1 0 0]
 [2 0 0]]

Coup 8
Règle 2 : pic 1 -> pic 2
Nouvelle situation : 196
[[3 0 0]
 [0 0 0]
 [2 1 0]]

Coup 9
Règle 1 : pic 0 -> pic 1
Nouvelle situation : 224
[[0 0 0]
 [3 0 0]
 [2 1 0]]

Coup 10
Règle 3 : pic 2 -> pic 1
Nouvelle situation : 168
[[0 0 0]
 [3 1 0]
 [2 0 0]]

Coup 11
Règle 4 : pic 1 -> pic 0
Nouvelle situation : 161
[[1 0 0]
 [3 0 0]
 [2 0 0]]

Coup 12
Règle 3 : pic 2 -> pic 1
Nouvelle situation : 49


## 14. Résultat de la résolution

À la fin du moteur, on vérifie si la situation finale a été atteinte et on affiche **le nombre de coups effectués**.

In [22]:
print("État final du jeu :")
print(jeu.pic)

print("\nNombre de palets sur chaque pic :")
print(jeu.nombre_palet)

État final du jeu :
[[0 0 0]
 [0 0 0]
 [3 2 1]]

Nombre de palets sur chaque pic :
[0 0 3]


In [23]:
if nombre_situation(jeu) == situation_finale:
    print("Situation finale atteinte.")
    print("Nombre de coups :", nombre_coup)
else:
    print("La situation finale n'a pas été atteinte.")
    print("Nombre de coups :", nombre_coup)

Situation finale atteinte.
Nombre de coups : 26


## 15. Analyse de la première résolution et limite du système

### Limite de la première résolution

Le système expert permet d’atteindre correctement la situation finale, mais la première exécution nécessite **26 coups**, alors que la solution optimale pour trois palets nécessite seulement **7 coups**.

Cette différence s’explique par la stratégie utilisée par le moteur d’inférence. Le moteur teste les règles dans l’ordre défini par la métarègle et applique la première règle éligible. Il cherche donc une solution valide en respectant les priorités de la base experte, mais **il ne recherche pas directement la solution comportant le nombre minimal de déplacements**.

La base experte initiale contient seulement quatre règles :

* Règle 1 : pic 0 → pic 1
* Règle 2 : pic 1 → pic 2
* Règle 3 : pic 2 → pic 1
* Règle 4 : pic 1 → pic 0

Cette limitation réduit les possibilités de déplacement direct du moteur et peut provoquer des déplacements supplémentaires.

### Proposition d’optimisation

Une première possibilité d'optimisation consiste à **enrichir la base experte avec de nouvelles règles de déplacement**, tout en conservant le même moteur d'inférence et la même métarègle.

On ajoute donc les deux déplacements directs qui ne figurent pas dans la base initiale :

* Règle 5 : pic 0 → pic 2
* Règle 6 : pic 2 → pic 0

Nous avons donc adopté une nouvelle méthode de sélection des règles. Au lieu de sélectionner systématiquement la première règle éligible selon l’ordre des indices, le moteur tient compte du numéro du coup et de la position du plus petit palet.

Pour les **coups impairs**, le plus petit palet est déplacé selon un cycle entre les trois pics :

`0 → 2 → 1 → 0`

Pour les **coups pairs**, le moteur recherche un déplacement légal d’un autre palet.

Cette stratégie utilise donc les six règles de la base experte, mais modifie la manière dont le moteur sélectionne la règle à appliquer. L'objectif est de guider davantage le chaînage avant vers une résolution minimale.

## 16. Base experte optimisée

On remplace maintenant la cellule de définition de `regles`. On introduit une methode de sélection qui force le déplacement du plus petit disque selon le cycle :
`0 → 2 → 1 → 0`, et aux coups pairs, choisit un autre déplacement légal.

In [24]:
# Base de règles améliorée
regles_ameliorees = [
    (0, 1),  # R1
    (1, 2),  # R2
    (2, 1),  # R3
    (1, 0),  # R4
    (0, 2),  # R5
    (2, 0)   # R6
]

In [25]:
def choisir_regle_amelioree(jeu, numero_coup):

    # Rechercher le pic contenant le plus petit palet (palet 1)
    position_petit = None

    for pic in range(3):
        for position in range(3):
            if jeu.pic[pic, position] == 1:
                position_petit = pic

    # Cycle de déplacement du plus petit palet :
    # 0 -> 2 -> 1 -> 0
    destination = {
        0: 2,
        2: 1,
        1: 0
    }

    # Coups impairs :
    # déplacer obligatoirement le plus petit palet
    if numero_coup % 2 == 1:

        pic_depart = position_petit
        pic_arrivee = destination[position_petit]

        for indice, (depart, arrivee) in enumerate(
            regles_ameliorees, start=1
        ):
            if depart == pic_depart and arrivee == pic_arrivee:

                if regle_jeu(depart, arrivee, jeu):
                    return indice

    # Coups pairs :
    # déplacer un palet différent du plus petit
    for indice, (depart, arrivee) in enumerate(
        regles_ameliorees, start=1
    ):

        if depart != position_petit:

            if regle_jeu(depart, arrivee, jeu):
                return indice

    return None

## 17. Moteur d’inférence avec la base optimisée

On repart d'un nouveau jeu initial, sinon le moteur continuerait à partir de l'état obtenu précédemment.

In [26]:
# Création de la situation finale
jeu_final = Jeu_Hanoi()

jeu_final.nombre_palet[2] = 3

jeu_final.pic[2, 0] = 3
jeu_final.pic[2, 1] = 2
jeu_final.pic[2, 2] = 1

# Identifiant de la situation finale
situation_finale = nombre_situation(jeu_final)

print("Identifiant de la situation finale :", situation_finale)

Identifiant de la situation finale : 448


In [27]:
# Réinitialisation du jeu
jeu = Jeu_Hanoi()

jeu.nombre_palet[0] = 3

jeu.pic[0, 0] = 3
jeu.pic[0, 1] = 2
jeu.pic[0, 2] = 1


# Initialisation
nbre_coup_ameliore = 0
historique_ameliore = []

print("Situation initiale :")
print(jeu.pic)
print()


# Moteur d'inférence optimisé
while nombre_situation(jeu) != situation_finale:
    nbre_coup_ameliore += 1

    regle = choisir_regle_amelioree(
        jeu,
        nbre_coup_ameliore
    )

    if regle is None:
        print("Aucune règle applicable.")
        break

    pic_depart, pic_arrivee = regles_ameliorees[regle - 1]

    effectue_deplacement(
        pic_depart,
        pic_arrivee,
        jeu
    )

    historique_ameliore.append(regle)

    print(
        f"Coup {nbre_coup_ameliore} : "
        f"R{regle} ({pic_depart} -> {pic_arrivee})"
    )

    print(jeu.pic)
    print()


print("Situation finale atteinte :", 
      nombre_situation(jeu) == situation_finale)

print("Nombre de coups après optimisation :", 
      nbre_coup_ameliore)

print("Règles utilisées :", 
      historique_ameliore)

print("\nÉtat final du jeu :")
print(jeu.pic)

print("\nNombre de palets sur chaque pic :")
print(jeu.nombre_palet)

Situation initiale :
[[3 2 1]
 [0 0 0]
 [0 0 0]]

Coup 1 : R5 (0 -> 2)
[[3 2 0]
 [0 0 0]
 [1 0 0]]

Coup 2 : R1 (0 -> 1)
[[3 0 0]
 [2 0 0]
 [1 0 0]]

Coup 3 : R3 (2 -> 1)
[[3 0 0]
 [2 1 0]
 [0 0 0]]

Coup 4 : R5 (0 -> 2)
[[0 0 0]
 [2 1 0]
 [3 0 0]]

Coup 5 : R4 (1 -> 0)
[[1 0 0]
 [2 0 0]
 [3 0 0]]

Coup 6 : R2 (1 -> 2)
[[1 0 0]
 [0 0 0]
 [3 2 0]]

Coup 7 : R5 (0 -> 2)
[[0 0 0]
 [0 0 0]
 [3 2 1]]

Situation finale atteinte : True
Nombre de coups après optimisation : 7
Règles utilisées : [5, 1, 3, 5, 4, 2, 5]

État final du jeu :
[[0 0 0]
 [0 0 0]
 [3 2 1]]

Nombre de palets sur chaque pic :
[0 0 3]



## 16. Conclusion et analyse finale des résultats

L’exécution du système expert optimisé permet d’atteindre la situation finale en **7 coups**.

La séquence obtenue est :

- **Coup 1 :** R5 - pic 0 → pic 2
- **Coup 2 :** R1 - pic 0 → pic 1
- **Coup 3 :** R3 - pic 2 → pic 1
- **Coup 4 :** R5 - pic 0 → pic 2
- **Coup 5 :** R4 - pic 1 → pic 0
- **Coup 6 :** R2 - pic 1 → pic 2
- **Coup 7 :** R5 - pic 0 → pic 2

La situation finale obtenue est bien :

```text
[[0 0 0]
 [0 0 0]
 [3 2 1]]
```

Tous les palets sont donc correctement transférés du pic 0 vers le pic 2.
L’optimisation permet donc de passer de **26 à 7 coups**, soit une réduction de **importante**. Le nombre obtenu correspond exactement à l’optimum théorique pour trois palets.

### Analyse

Ce résultat montre que la performance du système expert dépend fortement de la **base de règles** et surtout de la **méthode de sélection des règles**.

La première version, basée uniquement sur la priorité fixe des quatre règles initiales, permettait de trouver une solution valide mais non optimale. L’ajout des règles R5 et R6 augmente les possibilités de déplacement, mais ne suffit pas à lui seul à obtenir l’optimum avec la métarègle initiale.

La nouvelle méthode de sélection permet au contraire de guider le moteur d’inférence vers une séquence optimale. Dans notre expérimentation, le système atteint ainsi la solution en **7 coups**, sans utiliser l’algorithme récursif classique des Tours de Hanoï.

### Conclusion

Le système expert développé respecte les contraintes du problème et permet d’atteindre automatiquement la situation finale. L’expérimentation met également en évidence l’importance de la stratégie d’inférence : **une base de règles plus riche ne garantit pas à elle seule une résolution optimale ; la stratégie de sélection des règles joue également un rôle essentiel.**
